# Phase 4 — Narrative Extractor Raw Features

Tests N-01 through N-06.  

In [1]:
import sys, json, os
import numpy as np
import pandas as pd
from pathlib import Path
sys.path.insert(0, '/home/tamires/projects/rpp-aevans-ab/tamires/DecomposingMovies')
os.environ.pop('SSL_CERT_FILE', None)
os.environ['TF_CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

print(f"Python: {sys.version}")
print(f"NumPy:  {np.__version__}")

from cinematic_surprise.modalities.narrative import NarrativeExtractor
from cinematic_surprise.io.transcript import TranscriptReader
from cinematic_surprise.config import (
    TRANSCRIPT_COL_WORD, TRANSCRIPT_COL_START, TRANSCRIPT_COL_END, TRANSCRIPT_COL_TYPE
)

FIXTURES_DIR = Path("tests/fixtures/vectors")
FIXTURES_DIR.mkdir(parents=True, exist_ok=True)

# Path to Pulp Fiction subtitle CSV
# Adjust if your file is elsewhere
PF_CSV = Path("/home/tamires/projects/rpp-aevans-ab/tamires/audiovisual_stimuli/pulp_fiction_transcript.csv")

print(f"Pulp Fiction CSV: {PF_CSV} (exists: {PF_CSV.exists()})")

narrative_ext = NarrativeExtractor()
print("NarrativeExtractor loaded.")
print("\nSetup complete.")

Python: 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]
NumPy:  1.26.4


I0000 00:00:1774381755.398444 1303426 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774381755.442473 1303426 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774381756.434896 1303426 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Pulp Fiction CSV: /home/tamires/projects/rpp-aevans-ab/tamires/audiovisual_stimuli/pulp_fiction_transcript.csv (exists: True)


Loading weights: 100%|██████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 54293.49it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NarrativeExtractor loaded.

Setup complete.


---
## N-01: Word-to-Second Assignment
**Why:** Off-by-one puts words in the wrong second. Narrative surprise misaligns with fMRI.

In [2]:
if not PF_CSV.exists():
    print("N-01: SKIPPED — Pulp Fiction CSV not found")
else:
    df = pd.read_csv(PF_CSV)
    print(f"Loaded {len(df)} words from Pulp Fiction CSV")
    print(f"Columns: {list(df.columns)}")
    print(f"First 5 rows:")
    print(df.head())
    print()
    
    # Test: word at t=27.5s should go to second 27
    # From the CSV: 'forget' starts at 27.0, 'it' at 27.36, etc.
    
    # Group words by floor(start_time) = second assignment
    df['assigned_second'] = df[TRANSCRIPT_COL_START].apply(lambda x: int(np.floor(x)))
    
    # Check specific known words
    test_cases = [
        # (approximate start_time, expected_second, expected_word_fragment)
        (27.0, 27, "forget"),
        (27.36, 27, "it"),
        (29.5, 29, "im"),
        (30.07, 30, "shit"),
    ]
    
    all_ok = True
    for start_approx, expected_sec, expected_word in test_cases:
        # Find the word closest to this start time
        idx = (df[TRANSCRIPT_COL_START] - start_approx).abs().idxmin()
        row = df.loc[idx]
        actual_word = row[TRANSCRIPT_COL_WORD]
        actual_sec = row['assigned_second']
        
        ok = actual_sec == expected_sec
        if not ok:
            all_ok = False
        
        status = '✓' if ok else '✗'
        print(f"  t={row[TRANSCRIPT_COL_START]:.2f}s  word='{actual_word}'  "
              f"assigned=s{actual_sec}  expected=s{expected_sec}  {status}")
    
    # Edge case: word at exactly t=28.0
    print()
    edge_words = df[df[TRANSCRIPT_COL_START] == 28.0]
    if len(edge_words) > 0:
        edge_sec = int(np.floor(28.0))
        print(f"  Edge case t=28.0: assigned to second {edge_sec} (floor convention)")
    else:
        print(f"  No word at exactly t=28.0. Nearest:")
        near = df.iloc[(df[TRANSCRIPT_COL_START] - 28.0).abs().argsort()[:2]]
        for _, r in near.iterrows():
            print(f"    t={r[TRANSCRIPT_COL_START]:.2f}s  word='{r[TRANSCRIPT_COL_WORD]}'  "
                  f"assigned=s{r['assigned_second']}")
    
    # Check that floor() is the convention used
    # All words with start_time in [27.0, 28.0) should be in second 27
    sec27_words = df[(df[TRANSCRIPT_COL_START] >= 27.0) & (df[TRANSCRIPT_COL_START] < 28.0)]
    sec27_ok = all(sec27_words['assigned_second'] == 27)
    print(f"\n  All t∈[27,28) → second 27?  {sec27_ok}  {'✓' if sec27_ok else '✗'}")
    if not sec27_ok:
        all_ok = False
    
    print()
    if all_ok:
        print("N-01: PASSED ✓")
    else:
        print("N-01: FAILED ✗")

Loaded 15235 words from Pulp Fiction CSV
Columns: ['subtitle', 'start_time_new', 'end_time_new', 'interval_new', 'type']
First 5 rows:
  subtitle  start_time_new  end_time_new  interval_new     type
0   forget           27.00         27.36          0.36  matched
1       it           27.36         27.43          0.07  matched
2      its           27.44         27.56          0.12  matched
3      too           27.56         27.71          0.15  matched
4    risky           27.71         28.05          0.34  matched

  t=27.00s  word='forget'  assigned=s27  expected=s27  ✓
  t=27.36s  word='it'  assigned=s27  expected=s27  ✓
  t=29.50s  word='im'  assigned=s29  expected=s29  ✓
  t=30.07s  word='shit'  assigned=s30  expected=s30  ✓

  No word at exactly t=28.0. Nearest:
    t=27.71s  word='risky'  assigned=s27
    t=27.56s  word='too'  assigned=s27

  All t∈[27,28) → second 27?  True  ✓

N-01: PASSED ✓


---
## N-02: Embedding Determinism + Correct Dimension
**Why:** Non-deterministic embeddings = noise in narrative surprise. Wrong dim crashes.

In [4]:
test_sentence = "forget it its too risky"

# Embed twice
result_a = narrative_ext.extract(test_sentence)
result_b = narrative_ext.extract(test_sentence)

emb_a = result_a["narrative"]
emb_b = result_b["narrative"]

print(f"Sentence: '{test_sentence}'")
print(f"Shape A:  {emb_a.shape}")
print(f"Shape B:  {emb_b.shape}")
print(f"Dtype:    {emb_a.dtype}")
print(f"L2 norm:  {np.linalg.norm(emb_a):.4f}")
print()
identical = np.array_equal(emb_a, emb_b)
dim_ok = emb_a.ndim == 1 and emb_a.shape[0] == 384
non_zero = np.linalg.norm(emb_a) > 0.1
print(f"Bitwise identical?  {identical}  {'✓' if identical else '✗'}")
print(f"Shape (384,)?       {dim_ok}  {'✓' if dim_ok else '✗'}  (got {emb_a.shape})")
print(f"Non-zero?           {non_zero}  {'✓' if non_zero else '✗'}")
print()
if identical and dim_ok and non_zero:
    print("N-02: PASSED ✓")
else:
    if not identical:
        diff = np.abs(emb_a - emb_b).max()
        print(f"Max diff: {diff:.2e}")
    print("N-02: FAILED ✗")

Sentence: 'forget it its too risky'
Shape A:  (384,)
Shape B:  (384,)
Dtype:    float32
L2 norm:  1.0000

Bitwise identical?  True  ✓
Shape (384,)?       True  ✓  (got (384,))
Non-zero?           True  ✓

N-02: PASSED ✓


---
## N-03: Unpunctuated Sequences Are Non-Degenerate
**Why:** If all unpunctuated inputs collapse to similar vectors, narrative channel has no discriminative power.

In [12]:
def cosine_sim(a, b):
    a = a.ravel().astype(np.float64)
    b = b.ravel().astype(np.float64)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

# Two semantically different word sequences (no punctuation, as they come from the CSV)
sent_a = "forget it its too risky"
sent_b = "i know you killed him"
sent_c = "the weather is nice today"

emb_a = narrative_ext.extract(sent_a)["narrative"]
narrative_ext.reset()
emb_b = narrative_ext.extract(sent_b)["narrative"]
narrative_ext.reset()
emb_c = narrative_ext.extract(sent_c)["narrative"]
narrative_ext.reset()

sim_ab = cosine_sim(emb_a, emb_b)
sim_ac = cosine_sim(emb_a, emb_c)
sim_bc = cosine_sim(emb_b, emb_c)

print(f"'{sent_a}' vs '{sent_b}':         sim = {sim_ab:.4f}")
print(f"'{sent_a}' vs '{sent_c}':         sim = {sim_ac:.4f}")
print(f"'{sent_b}' vs '{sent_c}':         sim = {sim_bc:.4f}")
print()

# All pairs should have sim < 0.9 (they're semantically distinct)
all_distinct = sim_ab < 0.9 and sim_ac < 0.9 and sim_bc < 0.9
print(f"All pairs < 0.9?  {all_distinct}  {'✓' if all_distinct else '✗'}")
print()

if all_distinct:
    print("N-03: PASSED ✓")
else:
    print("N-03: FAILED ✗ — Embeddings are too similar for different sentences")

'forget it its too risky' vs 'i know you killed him':         sim = 0.1662
'forget it its too risky' vs 'the weather is nice today':         sim = -0.0228
'i know you killed him' vs 'the weather is nice today':         sim = 0.0820

All pairs < 0.9?  True  ✓

N-03: PASSED ✓


---
## N-04: Silence = Zero Embedding
**Why:** Stale embedding during silence means first-word-after-silence surprise spike is lost.

In [13]:
# The NarrativeExtractor is stateful by design:
# - Empty text AFTER speech → returns previous embedding (so EMA sees no change → zero surprise)
# - Empty text BEFORE any speech (fresh reset) → returns zero vector
# Both are correct. We test both.

narrative_ext.reset()

# Case 1: first call ever (after reset), empty string → should be zeros
emb_cold = narrative_ext.extract("")["narrative"]
print(f"Case 1 — Empty after reset (cold start):")
print(f"  Shape: {emb_cold.shape}")
print(f"  L2 norm: {np.linalg.norm(emb_cold):.6f}")
print(f"  Is zero: {np.allclose(emb_cold, 0.0)}")
cold_is_zero = np.allclose(emb_cold, 0.0)
print(f"  {'✓' if cold_is_zero else '✗'}")
print()

# Case 2: after real speech, empty string → should return previous embedding (NOT zero)
narrative_ext.reset()
real_emb = narrative_ext.extract("forget it its too risky")["narrative"]
emb_after_speech = narrative_ext.extract("")["narrative"]
print(f"Case 2 — Empty after speech:")
print(f"  Real embedding norm: {np.linalg.norm(real_emb):.4f}")
print(f"  Empty-after-speech norm: {np.linalg.norm(emb_after_speech):.4f}")

carries_forward = np.array_equal(emb_after_speech, real_emb)
print(f"  Carries forward previous? {carries_forward}")
print(f"  {'✓' if carries_forward else '✗'}")
print()

# Case 3: None input
narrative_ext.reset()
try:
    emb_none = narrative_ext.extract(None)["narrative"]
    print(f"Case 3 — None input:")
    print(f"  Shape: {emb_none.shape}")
    print(f"  Is zero: {np.allclose(emb_none, 0.0)}")
except Exception as e:
    print(f"Case 3 — None input raises: {type(e).__name__}: {e}")
    print("  (Acceptable if pipeline handles None before calling extract)")
print()

print("Summary:")
print(f"  Cold empty → zero?              {cold_is_zero}  {'✓' if cold_is_zero else '✗'}")
print(f"  Post-speech empty → carries fwd? {carries_forward}  {'✓' if carries_forward else '✗'}")
print()

if cold_is_zero and carries_forward:
    print("N-04: PASSED ✓")
    print("  The extractor correctly returns zeros before first speech,")
    print("  and carries forward the previous embedding during silence.")
    print("  This ensures EMA+KL sees no change during silence → zero surprise.")
else:
    print("N-04: FAILED ✗")

Case 1 — Empty after reset (cold start):
  Shape: (384,)
  L2 norm: 0.000000
  Is zero: True
  ✓

Case 2 — Empty after speech:
  Real embedding norm: 1.0000
  Empty-after-speech norm: 1.0000
  Carries forward previous? True
  ✓

Case 3 — None input raises: AttributeError: 'NoneType' object has no attribute 'strip'
  (Acceptable if pipeline handles None before calling extract)

Summary:
  Cold empty → zero?              True  ✓
  Post-speech empty → carries fwd? True  ✓

N-04: PASSED ✓
  The extractor correctly returns zeros before first speech,
  and carries forward the previous embedding during silence.
  This ensures EMA+KL sees no change during silence → zero surprise.


---
## N-05: Pin Sentence-Transformers Model Revision
**Why:** Model update shifts embedding space. EMA prior is now in wrong space.

In [9]:
import sentence_transformers

print(f"sentence-transformers version: {sentence_transformers.__version__}")

# Get model name/info
model_name = None
if hasattr(narrative_ext, 'model_name'):
    model_name = narrative_ext.model_name
elif hasattr(narrative_ext, 'model'):
    if hasattr(narrative_ext.model, 'model_card_data'):
        model_name = getattr(narrative_ext.model.model_card_data, 'model_name', None)
    if model_name is None and hasattr(narrative_ext.model, '_model_config'):
        model_name = getattr(narrative_ext.model._model_config, 'model_name', None)

print(f"Model name: {model_name}")

# Save pin
pin_file = FIXTURES_DIR / "narrative_pin.json"
pin_data = {
    "sentence_transformers_version": sentence_transformers.__version__,
    "model_name": str(model_name) if model_name else "unknown",
}

if pin_file.exists():
    with open(pin_file, "r") as f:
        saved = json.load(f)
    ok = True
    for k in pin_data:
        if k in saved and str(pin_data[k]) != str(saved[k]):
            print(f"  DRIFT: {k} saved={saved[k]} current={pin_data[k]}")
            ok = False
    if ok:
        print("Config matches saved pin. ✓")
    else:
        print("Config DRIFTED from pin. ✗")
else:
    with open(pin_file, "w") as f:
        json.dump(pin_data, f, indent=2)
    print(f"Saved pin to {pin_file}")
    ok = True

print()
if ok:
    print("N-05: PASSED ✓")
else:
    print("N-05: FAILED ✗")

sentence-transformers version: 5.3.0
Model name: None
Saved pin to tests/fixtures/vectors/narrative_pin.json

N-05: PASSED ✓


---
## N-06: Save Reference Embeddings
**Why:** Locks embedding space. Regression test catches model drift.

In [15]:
ref_file = FIXTURES_DIR / "narrative_reference_embeddings.npz"

test_sentences = [
    "forget it its too risky",
    "i know you killed him",
    "the weather is nice today",
    "we should go to the restaurant",
    "this is a very important moment",
]

narrative_ext.reset()  # clean state

current = {}
for i, sent in enumerate(test_sentences):
    emb = narrative_ext.extract(sent)["narrative"]
    current[f"sent_{i}"] = emb
    print(f"  sent_{i}: '{sent[:30]}...' shape={emb.shape} norm={np.linalg.norm(emb):.4f}")

if ref_file.exists():
    saved = dict(np.load(ref_file))
    ok = True
    for key in current:
        if key in saved:
            diff = np.abs(current[key] - saved[key]).max()
            match = diff < 1e-5
            status = '✓' if match else '✗'
            print(f"  {key}: max_diff = {diff:.2e}  {status}")
            if not match:
                ok = False
    print()
    if ok:
        print("N-06: PASSED ✓")
    else:
        print("N-06: FAILED ✗ — Embeddings differ from reference")
else:
    np.savez(ref_file, **current)
    print(f"\nSaved reference embeddings to {ref_file}")
    print("N-06: FIRST RUN — Re-run to verify.")

  sent_0: 'forget it its too risky...' shape=(384,) norm=1.0000
  sent_1: 'i know you killed him...' shape=(384,) norm=1.0000
  sent_2: 'the weather is nice today...' shape=(384,) norm=1.0000
  sent_3: 'we should go to the restaurant...' shape=(384,) norm=1.0000
  sent_4: 'this is a very important momen...' shape=(384,) norm=1.0000
  sent_0: max_diff = 0.00e+00  ✓
  sent_1: max_diff = 0.00e+00  ✓
  sent_2: max_diff = 0.00e+00  ✓
  sent_3: max_diff = 0.00e+00  ✓
  sent_4: max_diff = 0.00e+00  ✓

N-06: PASSED ✓
